In [1]:
import numpy as np
from sklearn.datasets import fetch_openml

In [2]:
# Load the full MNIST dataset
mnist = fetch_openml('mnist_784', as_frame=False, parser='auto')

# Load data and target variables
X, y = mnist.data, mnist.target
print(f"Data shape: {X.shape}, Target shape: {y.shape}")

# Turn into NumPy arrays
X = np.array(X)
y = np.array(y)

Data shape: (70000, 784), Target shape: (70000,)


In [3]:
# Create train and test sets
split_ratio = 0.9
train_size = int(split_ratio * len(X))

# Shuffle indices and reorder X and y
np.random.seed(46)
indices = np.arange(X.shape[0])
np.random.shuffle(indices)
X = X[indices]
y = y[indices]

# Split
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Transpose to (features, samples)
X_train = X_train.T
X_test = X_test.T

# Standardize
def standardize(X):
    mean = np.mean(X, 0)
    stdev = np.std(X, 0)
    return (X - mean) / stdev

X_train = standardize(X_train)
X_test = standardize(X_test)

# Ensure y is an integer array (sometimes fetch_openml returns strings)
y_train = y_train.astype(int)
y_test = y_test.astype(int)

In [4]:
def relu(Z):
    return np.maximum(0, Z)

def relu_backward(A, Z):
    return Z > 0

def softmax(Z):
    # axis=0 because each column is a sample
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True)) 
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

def softmax_backward_target(prediction, target):
    return prediction - target

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def sigmoid_backward(dA, Z):
    sig = sigmoid(Z)
    return dA * sig * (1 - sig)

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

In [5]:
def init_parameters(nn_architecture, seed=46):
    np.random.seed(seed)

    parameters = {}

    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        input_dim = layer["input_dim"]
        output_dim = layer["output_dim"]

        parameters['W' + str(layer_idx)] = np.random.randn(output_dim, input_dim) * 0.1
        parameters['b' + str(layer_idx)] = np.random.randn(output_dim, 1) * 0.1

    return parameters

def single_layer_forward(A_prev, W_curr, b_curr, activation="relu"):
    Z = np.dot(W_curr, A_prev) + b_curr

    if activation == "relu":
        A_curr = relu(Z)
    elif activation == "softmax":
        A_curr = softmax(Z)
    elif activation == "sigmoid":
        A_curr = sigmoid(Z)
    else:
        raise ValueError("Unsupported activation function")

    return Z, A_curr

def full_forward(X, parameters, nn_architecture):
    memory = {}
    A_curr = X

    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        A_prev = A_curr

        # Pass single layer forward
        curr_activation_function = layer['activation']
        W_curr = parameters["W" + str(layer_idx)]
        b_curr = parameters["b" + str(layer_idx)]
        A_curr, Z_curr = single_layer_forward(A_prev, W_curr, b_curr, curr_activation_function)

        # Save A_prev and Z_curr to memory because only A and Z
        # are needed to calculate dW and dB during the backwards pass
        memory["A" + str(idx)] = A_prev
        memory["Z" + str(layer_idx)] = Z_curr

    return A_curr, memory

def single_layer_backward(dA_curr, W_curr, b_curr, Z_curr, A_prev, activation="relu"):
    m = A_prev.size

    # dL/dZ_curr = backward_activation(A_curr, Z_curr)
    if activation == "relu":
        dZ_curr = relu_backward(dA_curr, Z_curr)
    elif activation == "sigmoid":
        dZ_curr = sigmoid_backward(dA_curr, Z_curr)
    elif activation == "softmax":
        dZ_curr = dA_curr # Handled in the full backward function with the softmax_backward_target function
    else:
        raise ValueError(f"Unsupported activation function {activation}")

    # dL/dW_curr = dL/dZ_curr * dZ_curr/dW_curr
    dW_curr = np.dot(dZ_curr, A_prev.T) * (1 / m)

    # dL/db_curr = dL/dZ_curr * dZ_curr/db_curr
    db_curr = np.sum(dZ_curr, axis=1, keepdims=True) * (1 / m)

    # dL/dA_prev = dL/dZ_curr * dZ_curr/dA_prev
    dA_prev = np.dot(W_curr.T, dZ_curr)

    return dA_prev, dW_curr, db_curr

def full_backward(prediction, target, memory, parameters, nn_architecture):
    gradients = {}
    m = target.size

    dA_prev = softmax_backward_target(prediction, one_hot(target))

    for layer_idx_prev, layer in reversed(list(enumerate(nn_architecture))):
        layer_idx_curr = layer_idx_prev + 1
        activation_function_curr = layer['activation']

        dA_curr = dA_prev

        A_prev = memory["A" + str(layer_idx_prev)]
        Z_curr = memory["Z" + str(layer_idx_curr)]
        W_curr = parameters["W" + str(layer_idx_curr)]
        b_curr = parameters["b" + str(layer_idx_curr)]

        dA_prev, dW_curr, db_curr = single_layer_backward(dA_curr, W_curr, b_curr, Z_curr, A_prev, activation_function_curr)

        gradients["dW" + str(layer_idx_curr)] = dW_curr
        gradients["db" + str(layer_idx_curr)] = db_curr

    return gradients

def update_parameters(parameters, gradients, nn_architecture, learning_rate=0.01):
    for idx, layer in enumerate(nn_architecture):
        layer_idx = idx + 1
        parameters["W" + str(layer_idx)] = parameters["W" + str(layer_idx)] - learning_rate * gradients["dW" + str(layer_idx)]
        parameters["b" + str(layer_idx)] = parameters["b" + str(layer_idx)] - learning_rate * gradients["db" + str(layer_idx)]

    return parameters

In [6]:
def get_accuracy(predictions, target):
    return np.sum(predictions == target) / target.size

def get_cost(prediction, target):
    # Categorical cross-entropy cost function
    m = target.shape[1]
    prediction = np.clip(prediction, 1e-15, 1 - 1e-15)
    
    return -1 / m * np.sum(target * np.log(prediction))

def train(X, Y , nn_architecture, epochs, learning_rate):
    parameters = init_parameters(nn_architecture, seed=46)
    cost_history = []
    accuracy_history = []

    for i in range(epochs):
        # Pass forward through the model
        prediction, cache = full_forward(X, parameters, nn_architecture)

        # Save cost and accuracy history
        cost_history.append(get_cost(prediction, one_hot(Y)))
        accuracy_history.append(get_accuracy(np.argmax(prediction, 0), Y))

        gradients = full_backward(prediction, Y, cache, parameters, nn_architecture)
        parameters = update_parameters(parameters, gradients, nn_architecture, learning_rate)

        if i % 10 == 0:
            print(f"---- Epoch {i}\nCost = {cost_history[-1]:.4f}, Accuracy = {accuracy_history[-1]:.4f}")

    return parameters, cost_history, accuracy_history

In [8]:
nn_architecture = [
    {"input_dim": 784, "output_dim": 64, "activation": "relu"},
    {"input_dim": 64, "output_dim": 10, "activation": "softmax"}
]

parameters, cost_history, accuracy_history = train(X_train, y_train, nn_architecture, 1500, 3e-2)

---- Epoch 0
Cost = 19.7078, Accuracy = 0.1316
---- Epoch 10
Cost = 18.2028, Accuracy = 0.1289
---- Epoch 20
Cost = 17.1911, Accuracy = 0.1268
---- Epoch 30
Cost = 16.4634, Accuracy = 0.1267
---- Epoch 40
Cost = 15.9015, Accuracy = 0.1282
---- Epoch 50
Cost = 15.4363, Accuracy = 0.1324
---- Epoch 60
Cost = 15.0258, Accuracy = 0.1376
---- Epoch 70
Cost = 14.6858, Accuracy = 0.1429
---- Epoch 80
Cost = 14.3378, Accuracy = 0.1475
---- Epoch 90
Cost = 14.0284, Accuracy = 0.1523
---- Epoch 100
Cost = 13.7311, Accuracy = 0.1566
---- Epoch 110
Cost = 13.4518, Accuracy = 0.1605
---- Epoch 120
Cost = 13.1919, Accuracy = 0.1640
---- Epoch 130
Cost = 12.9335, Accuracy = 0.1671
---- Epoch 140
Cost = 12.6836, Accuracy = 0.1707
---- Epoch 150
Cost = 12.4734, Accuracy = 0.1738
---- Epoch 160
Cost = 12.2458, Accuracy = 0.1772
---- Epoch 170
Cost = 12.0426, Accuracy = 0.1800
---- Epoch 180
Cost = 11.8362, Accuracy = 0.1833
---- Epoch 190
Cost = 11.6513, Accuracy = 0.1862
---- Epoch 200
Cost = 11.4666, 